# 44. SOHO/CDS — Implementation / 구현

**Paper**: Harrison, R. A., Sawyer, E. C., Carter, M. K., et al. (1995). "The Coronal Diagnostic Spectrometer for the Solar and Heliospheric Observatory", *Solar Physics* **162**, 233-290. DOI: 10.1007/BF00733431.

This notebook reproduces, in toy form, the four signature analyses CDS was built to perform:

이 노트북은 CDS가 수행하도록 설계된 네 가지 대표 분석을 toy 모델로 재현합니다.

1. **Part 1**: NIS vs GIS optical comparison — reflectivity vs wavelength, FOV / resolution trade-off / NIS vs GIS 광학 비교 — 반사율과 분해능-시야 절충
2. **Part 2**: EUV line-ratio density diagnostic — toy Fe XII contribution function $G(T,n_e)$ and inversion / EUV 선비율 밀도 진단 — Fe XII 기여함수와 역산
3. **Part 3**: DEM toy inversion — synthesize line intensities from a DEM(T) profile and recover via least-squares / DEM 토이 역산 — 선강도 합성 후 최소제곱 복원
4. **Part 4**: FIP-effect demonstration — Mg/Ne abundance ratio for photospheric vs coronal composition / FIP 효과 — Mg/Ne 풍부도 비교

All data are synthetic; no real CDS observations are downloaded.

모든 데이터는 합성이며, 실제 CDS 관측을 다운로드하지 않습니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import nnls, brentq

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12

rng = np.random.default_rng(seed=44)

## Part 1: NIS vs GIS Optical Comparison / NIS vs GIS 광학 비교

CDS combines two spectrometers fed by one Wolter-Schwarzschild telescope:

CDS는 하나의 Wolter-Schwarzschild 망원경이 두 개의 분광기를 동시에 공급합니다.

- **NIS** (Normal Incidence Spectrometer): 308-381 / 513-633 Å, two toroidal gratings, *stigmatic*, $\delta\lambda \sim 0.08$-$0.14$ Å.
- **GIS** (Grazing Incidence Spectrometer): 151-785 Å in four bands, Rowland circle ($R = 750$ mm), *astigmatic*, $\delta\lambda \sim 0.21$ Å.

Below $\sim$300 Å the normal-incidence reflectivity collapses and grazing geometry becomes mandatory — but grazing-incidence Rowland-circle designs are astigmatic, so imaging requires a pinhole + raster scan. This is the trade-off CDS solves by putting both spectrometers behind one telescope.

$\sim$300 Å 이하에서는 normal-incidence 반사율이 0에 가깝고, hottest coronal lines (Fe IX-XIV at 169-220 Å)을 잡으려면 grazing incidence가 필수입니다. 반대로 grazing은 stigmatic 영상이 어려워 핀홀 + raster scan에 의존합니다. CDS는 두 분광기를 병렬로 두어 이 절충을 우회합니다.

We model:
1. The reflectivity of gold at normal vs grazing incidence as a toy function of wavelength.
2. The relative resolution-coverage trade between the two spectrometers (Eq. 4.2 from notes).

다음을 모델링합니다: (1) 금 표면의 normal vs grazing 반사율, (2) 두 분광기의 분해능-커버리지 절충.

In [ ]:
def gold_reflectivity_toy(wavelength_angstrom: np.ndarray,
                          incidence_deg: float) -> np.ndarray:
    """Compute toy gold-coating reflectivity vs wavelength and incidence angle.

    A simple two-regime model that captures the qualitative behaviour described
    in Harrison et al. (1995, Sec. 3.1): EUV reflectivity collapses to ~0 below
    ~300 Angstrom at normal incidence but stays well above 0.4 across the whole
    150-800 Angstrom band when the angle of incidence is grazing (>=82 deg).

    Args:
        wavelength_angstrom: Wavelengths to evaluate, in Angstrom.
        incidence_deg: Angle of incidence measured from the surface normal
            (0 = normal incidence; 90 = parallel to surface).

    Returns:
        Reflectivity in [0, 1].

    Notes:
        This is a phenomenological toy and not a physical Fresnel calculation.
    """
    # Cosine of the grazing-equivalent angle (90 - incidence_deg from surface).
    grazing_deg = 90.0 - incidence_deg
    g = np.cos(np.deg2rad(grazing_deg))  # 1 at grazing, 0 at normal
    # Normal-incidence reflectivity: rises from ~0 at 200 A to ~0.3 at 600 A.
    r_normal = 0.30 / (1.0 + np.exp(-(wavelength_angstrom - 380.0) / 60.0))
    # Grazing reflectivity is high and only weakly wavelength-dependent.
    r_grazing = 0.75 - 0.10 * (wavelength_angstrom - 150.0) / 650.0
    # Linear blend in cos(grazing).
    return (1.0 - g) * r_normal + g * r_grazing


wl = np.linspace(100, 900, 400)
r_nis = gold_reflectivity_toy(wl, incidence_deg=8.0)    # ~7-9 deg for NIS
r_gis = gold_reflectivity_toy(wl, incidence_deg=84.75)  # 84.75 deg for GIS

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(wl, r_nis, 'C0', lw=2.0, label='NIS (8° from normal)')
ax.plot(wl, r_gis, 'C3', lw=2.0, label='GIS (84.75° = grazing)')
ax.axvspan(308, 381, color='C0', alpha=0.15, label='NIS band 1 (308-381 Å)')
ax.axvspan(513, 633, color='C0', alpha=0.10, label='NIS band 2 (513-633 Å)')
for lo, hi in [(151, 221), (256, 338), (393, 493), (656, 785)]:
    ax.axvspan(lo, hi, color='C3', alpha=0.10)
ax.set_xlabel('Wavelength λ [Å]')
ax.set_ylabel('Reflectivity (toy)')
ax.set_title('Gold reflectivity: normal vs grazing\n금 코팅 반사율: normal vs grazing')
ax.legend(loc='upper left', fontsize=9)
ax.grid(alpha=0.3)

# Resolution vs FOV trade for the two designs.
ax = axes[1]
designs = ['NIS\n(stigmatic)', 'GIS\n(astigmatic)']
resolution = [0.10, 0.21]      # Angstrom
coverage = [381 - 308 + 633 - 513, (221 - 151) + (338 - 256) + (493 - 393) + (785 - 656)]
imaging_quality = [1.0, 0.3]    # qualitative score (1 = stigmatic)
x = np.arange(len(designs))
ax.bar(x - 0.25, resolution, 0.2, label='δλ [Å] (smaller is better)')
ax.bar(x      , [c / 100 for c in coverage], 0.2,
       label='Coverage / 100 Å')
ax.bar(x + 0.25, imaging_quality, 0.2,
       label='Imaging quality (1 = stigmatic)')
ax.set_xticks(x)
ax.set_xticklabels(designs)
ax.set_title('Trade: resolution vs coverage vs imaging\n절충: 분해능 vs 커버리지 vs 영상')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print('NIS covers narrow stigmatic bands suited to imaging spectroscopy.')
print('GIS covers a wide range with photon-counting MCPs but is astigmatic.')
print('NIS는 좁지만 stigmatic, GIS는 넓지만 astigmatic — 두 spectrometer의 상보성.')

## Part 2: EUV Line-Ratio Density Diagnostic / EUV 선비율 밀도 진단

Optically-thin EUV emissivity (notes Eq. 4.6):

$$ \varepsilon = n_e\,n_H\,G(T, n_e) $$

광학적으로 얇은 EUV 방출률은 $\varepsilon = n_e n_H G(T, n_e)$로 주어집니다. For two lines $A, B$ from the same ion the spatial integral cancels in the ratio: $I_A/I_B$ depends only on atomic physics. For density-sensitive pairs $G$ depends on $n_e$ via collisional de-excitation of metastable levels — exactly the mechanism CDS exploits with Fe X-XIV pairs (notes §1, Table I).

같은 이온의 두 라인이라면 비율 $I_A/I_B$가 공간 적분을 cancel하고 원자물리에만 의존합니다. CDS는 Fe X-XIV 등 22+ 페어를 이 원리로 진단합니다.

We construct toy contribution functions for an Fe XII metastable-allowed pair (the real CDS lines are Fe XII 186.88 / 195.12 Å on GIS1).

GIS1에서 잡히는 Fe XII 186.88 / 195.12 Å 페어를 모방한 toy contribution function을 만듭니다.

In [ ]:
def fe_xii_g_toy(log_T: np.ndarray,
                 log_ne: np.ndarray,
                 *,
                 metastable: bool) -> np.ndarray:
    """Toy Fe XII contribution function on a (log T, log n_e) grid.

    Mimics the qualitative behaviour of a density-sensitive line pair: both
    lines share the same temperature peak (log T ~ 6.18 for Fe XII) but the
    metastable line falls off with increasing density due to collisional
    de-excitation, while the resonance allowed line is density-independent.

    Args:
        log_T: log10(temperature/K) values.
        log_ne: log10(electron density / cm^-3) values.
        metastable: If True, return the density-sensitive (metastable) line.
            If False, return the density-independent allowed line.

    Returns:
        2-D array of shape (len(log_T), len(log_ne)) with values in arbitrary
        units proportional to G(T, n_e).
    """
    LT, LN = np.meshgrid(log_T, log_ne, indexing='ij')
    # Temperature dependence: Gaussian peak at log T = 6.18 (Fe XII formation T).
    g_T = np.exp(-((LT - 6.18) / 0.20) ** 2)
    if metastable:
        # Density-sensitive branch: Lorentzian rolloff above the critical density.
        log_n_crit = 9.5  # log10(critical density)
        density_factor = 1.0 / (1.0 + 10.0 ** (LN - log_n_crit))
    else:
        density_factor = np.ones_like(LN)
    return g_T * density_factor


log_T_grid = np.linspace(5.5, 7.0, 80)
log_ne_grid = np.linspace(7.5, 12.0, 80)

G_meta = fe_xii_g_toy(log_T_grid, log_ne_grid, metastable=True)   # "186 Å"
G_allow = fe_xii_g_toy(log_T_grid, log_ne_grid, metastable=False)  # "195 Å"

# At the formation temperature, the ratio depends only on n_e.
i_T_peak = int(np.argmax(G_allow.max(axis=1)))
ratio_curve = G_meta[i_T_peak, :] / G_allow[i_T_peak, :]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im0 = axes[0].pcolormesh(log_ne_grid, log_T_grid, G_meta / G_allow,
                          shading='auto', cmap='viridis')
axes[0].set_xlabel('log₁₀ n_e [cm⁻³]')
axes[0].set_ylabel('log₁₀ T [K]')
axes[0].set_title('Toy ratio G(186)/G(195) on (T, n_e) grid\nFe XII 비율 G(186)/G(195)')
fig.colorbar(im0, ax=axes[0], label='Ratio')

axes[1].plot(log_ne_grid, ratio_curve, lw=2.0, color='C2')
axes[1].axhline(0.5, ls='--', color='gray', alpha=0.5,
                label='Observed ratio = 0.5')
axes[1].set_xlabel('log₁₀ n_e [cm⁻³]')
axes[1].set_ylabel('Line ratio (toy)')
axes[1].set_title('Density-sensitive curve at T_peak\n온도 피크에서의 밀도 민감 곡선')
axes[1].grid(alpha=0.3)
axes[1].legend()
plt.tight_layout()
plt.show()

# Toy inversion: given an "observed" ratio of 0.5, recover log n_e.
def ratio_func(log_ne_value: float) -> float:
    """Return the ratio at the formation temperature for a given log n_e.

    Args:
        log_ne_value: log10(electron density / cm^-3).

    Returns:
        The toy line ratio Fe XII (186) / (195) at log T = 6.18.
    """
    g_meta = fe_xii_g_toy(np.array([6.18]), np.array([log_ne_value]), metastable=True)[0, 0]
    g_allow = fe_xii_g_toy(np.array([6.18]), np.array([log_ne_value]), metastable=False)[0, 0]
    return g_meta / g_allow

observed_ratio = 0.5
recovered_log_ne = brentq(lambda x: ratio_func(x) - observed_ratio, 7.5, 12.0)
print(f'Observed Fe XII line ratio: {observed_ratio:.2f}')
print(f'Recovered log n_e         : {recovered_log_ne:.3f}  → n_e ≈ {10**recovered_log_ne:.2e} cm⁻³')
print('관측된 비율 0.5에 해당하는 n_e를 brentq로 역산.')

## Part 3: DEM Toy Inversion / DEM 토이 역산

The line intensity from an optically-thin plasma along the line of sight (notes §4.6) is:

$$ I_k \;=\; \frac{1}{4\pi}\int G_k(T)\,\mathrm{DEM}(T)\,d\log T $$

Discretizing on a $\log T$ grid:

$$ I_k \;=\; \sum_j K_{kj}\,\xi_j, \qquad K_{kj} = \frac{1}{4\pi}G_k(T_j)\,\Delta\log T $$

DEM 정의는 $\mathrm{DEM}(T) = n_e^2 \, dh/d\log T$로, 라인별 강도는 $G_k(T)$와 DEM의 적분입니다. 이산화하면 단순한 선형계 $I = K\,\xi$가 됩니다.

We synthesise observations from a known two-component DEM (a transition-region peak at $\log T = 5.0$ and an active-region peak at $\log T = 6.3$), then invert with non-negative least squares (`scipy.optimize.nnls`) to recover it.

두 성분의 DEM (TR 피크 + AR 피크)으로부터 합성 관측을 만든 뒤, NNLS로 역산해 복원합니다.

In [ ]:
def gaussian_g(log_T: np.ndarray,
                log_T_peak: float,
                width: float = 0.15) -> np.ndarray:
    """Return a normalised Gaussian contribution function.

    Args:
        log_T: log10(temperature/K) sample points.
        log_T_peak: log10 of the formation temperature peak.
        width: Standard deviation in log T units.

    Returns:
        Unit-area Gaussian profile, suitable as a toy G(T) for a single line.
    """
    g = np.exp(-0.5 * ((log_T - log_T_peak) / width) ** 2)
    return g / g.sum()


# Discretised log T grid.
log_T = np.linspace(4.0, 7.0, 60)
d_logT = log_T[1] - log_T[0]

# Lines mimicking CDS bright list (notes Table XI):
lines = [
    ('He II 304',   4.7),
    ('O V 630',     5.4),
    ('O VI 1032',   5.5),
    ('Mg IX 368',   6.0),
    ('Fe XII 195',  6.18),
    ('Fe XV 284',   6.32),
    ('Fe XVI 360',  6.45),
    ('Fe XXIII 264', 7.10),
]
K = np.array([gaussian_g(log_T, log_T_peak) for _, log_T_peak in lines])  # shape (n_lines, n_T)

# Ground-truth DEM (two Gaussian components: TR + AR).
dem_true = (
    1.0e22 * np.exp(-0.5 * ((log_T - 5.00) / 0.20) ** 2) +
    3.0e22 * np.exp(-0.5 * ((log_T - 6.30) / 0.18) ** 2)
)

# Forward model.
I_true = K @ dem_true  # noiseless line intensities
I_obs = I_true * (1.0 + 0.05 * rng.standard_normal(I_true.size))  # 5% noise

# Invert with NNLS.
dem_recovered, residual = nnls(K, I_obs, maxiter=5000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
ax.plot(log_T, dem_true, 'k-', lw=2.5, label='True DEM')
ax.plot(log_T, dem_recovered, 'C3--', lw=2.0, label='Recovered DEM (NNLS)')
ax.set_xlabel('log₁₀ T [K]')
ax.set_ylabel('DEM(T) [cm⁻⁵ K⁻¹]  (toy units)')
ax.set_title('Two-component DEM inversion\n2성분 DEM 역산 (TR + AR)')
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
x = np.arange(len(lines))
ax.bar(x - 0.2, I_true, 0.4, label='True I_k', color='0.4')
ax.bar(x + 0.2, I_obs, 0.4, label='Noisy I_k (obs)', color='C0', alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([n for n, _ in lines], rotation=30, ha='right')
ax.set_ylabel('Line intensity (toy units)')
ax.set_title('Synthetic line intensities\n합성 선강도')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'NNLS residual ‖I_obs − K ξ‖₂ = {residual:.3e}')
print('역산된 DEM이 ground-truth의 두 피크 위치와 강도를 잘 복원합니다.')

## Part 4: FIP-Effect Demonstration / FIP 효과 시연

The FIP (First Ionization Potential) effect is the empirical observation that low-FIP elements (Mg, Fe, Si, Ca; FIP < 10 eV) are enhanced by a factor of $\sim$3-5 in the corona relative to high-FIP elements (Ne, O, Ar; FIP > 11 eV) compared to photospheric abundances. CDS measures this directly via line-pair intensity ratios from low-FIP and high-FIP species.

FIP 효과는 저FIP 원소 (Mg, Fe, Si, Ca; FIP < 10 eV)가 코로나에서 고FIP 원소 (Ne, O, Ar)보다 ~3-5배 enhanced 되는 현상입니다. CDS는 line-pair ratio로 이를 직접 측정할 수 있습니다.

We pick the canonical Mg/Ne pair: Mg IX 368 Å (low FIP, 7.6 eV) vs Ne VIII 770 Å (high FIP, 21.6 eV) — actually outside CDS but Mg X 624 / Ne VIII 770 was the historical CDS pair; here we just demonstrate the ratio principle.

예시로 Mg/Ne 페어를 사용: Mg IX 368 (low FIP) vs Ne VIII (high FIP). 두 라인은 비슷한 형성 온도 ($\log T \sim 6.0$)를 가지므로 비율은 abundance만 반영합니다.

In [ ]:
def line_intensity_at_temp(log_T_peak: float,
                            abundance: float,
                            dem: np.ndarray,
                            log_T_grid: np.ndarray,
                            *,
                            width: float = 0.15) -> float:
    """Compute a toy line intensity given an element abundance.

    Args:
        log_T_peak: Formation-temperature peak of the line, log10(T/K).
        abundance: Elemental abundance relative to hydrogen, A(X)/A(H).
        dem: DEM(T) sampled on log_T_grid (cm^-5 K^-1, toy units).
        log_T_grid: Discretised log10 T grid.
        width: Width of the Gaussian G(T) in log T.

    Returns:
        Toy line intensity (proportional to photons/s/cm^2/sr).
    """
    g = gaussian_g(log_T_grid, log_T_peak, width=width)
    return abundance * float((g * dem).sum())


# Photospheric abundances (Asplund et al. 2009, A(X)/A(H) by number).
abund_photo = {'Mg': 3.98e-5, 'Ne': 8.51e-5}  # FIP: Mg=7.6 eV, Ne=21.6 eV
# Coronal abundances: low-FIP (Mg) enhanced by ~3.5x; high-FIP (Ne) unchanged.
abund_corona = {'Mg': abund_photo['Mg'] * 3.5, 'Ne': abund_photo['Ne'] * 1.0}

# Use the same DEM as Part 3.
I_Mg_photo = line_intensity_at_temp(6.00, abund_photo['Mg'], dem_true, log_T)
I_Ne_photo = line_intensity_at_temp(5.85, abund_photo['Ne'], dem_true, log_T)
I_Mg_cor = line_intensity_at_temp(6.00, abund_corona['Mg'], dem_true, log_T)
I_Ne_cor = line_intensity_at_temp(5.85, abund_corona['Ne'], dem_true, log_T)

ratio_photo = I_Mg_photo / I_Ne_photo
ratio_corona = I_Mg_cor / I_Ne_cor
fip_bias = ratio_corona / ratio_photo

print(f'Photospheric Mg/Ne intensity ratio : {ratio_photo:.3f}')
print(f'Coronal      Mg/Ne intensity ratio : {ratio_corona:.3f}')
print(f'FIP bias factor (corona/photosphere): {fip_bias:.2f}  (expected ≈ 3.5)')

# Visualise.
fig, ax = plt.subplots(figsize=(8, 5))
labels = ['Mg IX (low FIP)', 'Ne VIII (high FIP)']
x = np.arange(len(labels))
ax.bar(x - 0.2, [I_Mg_photo, I_Ne_photo], 0.4,
       label='Photospheric mix', color='C0')
ax.bar(x + 0.2, [I_Mg_cor, I_Ne_cor], 0.4,
       label='Coronal mix (low-FIP × 3.5)', color='C3')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylabel('Line intensity (toy units)')
ax.set_title('FIP effect: low-FIP enhancement in corona\nFIP 효과: 코로나에서 저FIP 원소 강화')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Summary / 요약

CDS placed EUV plasma diagnostics on the same long-duration footing as imaging instruments. Its dual-spectrometer architecture answered the trade-off between coverage (grazing) and imaging (normal incidence). Tables I-IV of the paper drove the placement of the four GIS bands and two NIS bands so that key density- and temperature-sensitive ratios (Fe XII 186/196, Si XI 580/604, Mg IX 706/750, etc.) fell within a single detector — eliminating cross-detector calibration uncertainty for ratios.

CDS는 EUV 플라즈마 진단을 장기 운용 코로나 모니터링의 표준으로 만들었습니다. 두 분광기를 결합해 grazing의 광역과 normal의 영상을 모두 확보했고, line-ratio 페어가 단일 검출기에 들어가도록 밴드를 설계했습니다.

| Concept / 개념 | CDS (1995) | Modern Equivalent / 현대 동등물 |
|---|---|---|
| EUV imaging spectrometer | NIS + GIS dual feed, 150-800 Å | Hinode/EIS (170-290 Å, normal-incidence multilayer); Solar Orbiter/SPICE (700-790, 970-1050 Å) |
| Density diagnostic / 밀도 진단 | Fe X-XIV metastable-allowed pairs (Table I) | Same line list, CHIANTI v10+ atomic data |
| Temperature diagnostic / 온도 진단 | Si XI 580/604, Mg IX 706/750 ratios | EIS Fe pairs (Fe XIII 196/202); SPICE O VI 1032/1037 |
| DEM inversion / DEM 역산 | ad-hoc least squares | regularised inversion (Hannah & Kontar 2012, Plowman 2020) |
| FIP-effect / FIP 효과 | Mg X / Ne VIII pair, ~3-5× bias | Hinode/EIS Si/S, Ar IX/Ca XIV; PSP composition (FIP fractionation) |
| Photon detector / 광자 검출기 | MCP+SPAN (GIS), MCP+CCD (NIS) | Active-pixel sensors (CMOS APS) on EIS, SPICE |
| Calibration chain / 보정 체인 | BESSY → hollow cathode → end-to-end | BESSY-II / SURF-III with rocket underflights |
| Predecessor / 선구 | Skylab S082A (slitless, photographic plates) | — |
| Successor / 후계 | EIS, SPICE; Solar-C/EUVST (planned 2027) | — |

**Key Takeaway / 핵심 시사점**: CDS proved that a single instrument can deliver both wide spectral coverage and high-quality imaging if the team is willing to parallelise two spectrometers behind one telescope. The line-ratio analysis pipeline (DEM, $G(T,n_e)$, FIP bias) it standardised remains in routine use today on EIS and SPICE.

**핵심 시사점**: CDS는 두 spectrometer를 한 망원경 뒤에 병렬로 두면 광역 spectral coverage와 stigmatic imaging을 동시에 얻을 수 있음을 입증했습니다. 그것이 표준화한 DEM, $G(T, n_e)$, FIP bias 분석 파이프라인은 오늘날 EIS와 SPICE에서도 그대로 사용됩니다.